# PQC 的 barren plateau：测量梯度分布，而不是只画一条损失线

当随机初始化后的梯度集中在零附近，优化可能需要更多精度与测量资源。这里固定一个 `qnn.PQC` 电路结构，在不同量子比特数、深度和初始化 seed 下测量**同一个参数**的梯度。

我们对比同为 `[-1, 1]` 范围的局部 cost `⟨Z₀⟩` 和全局 cost `⟨Z₀⊗⋯⊗Zₙ₋₁⟩`，没有人为乘上随比特数缩小的系数。模拟使用精确期望值，图中的随机性来自参数初始化，不是硬件 shot noise。

相关理论有电路分布、design 和局部性等条件；本小规模、固定结构实验不自动满足这些条件，也不证明所有 PQC 都有指数梯度衰减。参见 [McClean et al. (2018)](https://arxiv.org/abs/1803.11173) 与 [Cerezo et al. (2021)](https://arxiv.org/abs/2001.00550)。

In [ ]:
import sys

# 支持项目根、examples、当前quantum目录作为工作目录。
sys.path[:0] = ["examples/quantum", "quantum", "."]

from math import log10
from qutai import qnn, visualization as vis
from barren_plateau import collect, gradient_pair, summarize, variance_plot

device = "cuda"  # 无CUDA时改成 "cpu"，也可先减少samples。
dtype = "float32"
qubit_counts = (2, 4, 6, 8)
depths = (1, 4)
samples = 24


## 1. 固定结构，重采样参数

PQC 的 RX/RY/RZ 结构固定为 `seed=23`，纠缠门为相邻 CZ；每次只改变角度。选中的参数是第一层 q0 的 RY 角度 `θ[0,0]`，它不会因为最初位于零态而成为恒不可见的 RZ。

`wide` 从 `[-π,π]` 采样，`small` 从 `[-0.1,0.1]` 采样。两种 cost 共享相同参数点；不同组使用相同 seed 便于配对比较。SDK 的固定读出旋转也属于电路结构，小角度不代表整段电路是恒等门。

In [ ]:
pqc = qnn.PQC(4, depth=4, entangler="cz", seed=23).to(device, dtype=dtype)
print("固定旋转轴：", pqc.rotations)
print("参数形状：", pqc.weight.shape)
vis.Circuit(pqc.circuit([0.0] * 4), title="固定结构的4比特PQC")


## 2. 先核对求导，再测分布

小电路用 CPU/float64 对照伴随法与参数移位。完整实验仍在上方选定设备执行。

另外加入无纠缠的 `RY` 乘积态对照。若角度独立均匀分布于 `[-π,π]`，可直接算出局部梯度方差为 `1/2`，全局梯度方差为 `2⁻ⁿ`。这提醒我们：全局 cost 的梯度衰减并不必然来自纠缠。有限样本估计不需要恰好等于理论值。

In [ ]:
adjoint = gradient_pair(3, 2, 1003, device="cpu", dtype="float64", method="adjoint")
shift = gradient_pair(3, 2, 1003, device="cpu", dtype="float64", method="parameter-shift")
gradient_error = max(abs(adjoint[name] - shift[name]) for name in ("local_gradient", "global_gradient"))
print("伴随/参数移位最大梯度差：", gradient_error)
assert gradient_error < 1e-10
assert abs(adjoint["local_gradient"]) + abs(adjoint["global_gradient"]) > 1e-8


## 3. 多 seed 实际采样

默认 2/4/6/8 个比特、深度 1/4，每组 24 个独立参数 seed。每个参数点都实际前向并分别对两种 cost 反向；不会用理论曲线替换测量值。完整函数在同目录 [barren_plateau.py](barren_plateau.py)，只调用 SDK 公共接口。

样本方差采用 `N-1` 分母；95% 区间来自 400 次 percentile bootstrap，仅反映有限初始化样本的不确定性。增加比特数时，状态向量仍有 `2**qubits` 个振幅。

In [ ]:
records = collect(qubit_counts, depths, samples, device=device, dtype=dtype)
bp_summary = summarize(records)
factorized_error = max(row.get("analytic_gradient_max_abs", 0.0) for row in records)
print("实际参数点数：", len(records), "梯度读出数：", 2 * len(records))
print("无纠缠对照：量子反向/解析梯度最大差：", factorized_error)
assert factorized_error < (2e-5 if dtype == "float32" else 1e-10)
assert all(row["variance"] >= 0 for row in bp_summary)


In [ ]:
vis.Table(
    [[r["qubits"], r["depth"], r["initialization"], r["cost"],
      f'{r["mean"]:.3e}', f'{r["rms"]:.3e}', f'{r["variance"]:.3e}',
      f'[{r["ci_low"]:.2e}, {r["ci_high"]:.2e}]']
     for r in bp_summary if r["kind"] == "pqc"],
    columns=["比特", "深度", "初始化", "cost", "梯度均值", "梯度RMS", "样本方差", "95% bootstrap区间"],
    title="固定PQC的实测统计（保留均值，避免把近似常数的大梯度误判为平台）",
)


## 4. 看方差，也看原始梯度

曲线展示宽随机初始化的 `log10(样本方差)`，零值仅在绘图时截到 `1e-30`，表格保留原值。少量比特和有限 seed 不要求结果单调；误差区间可能很宽。

第二张图展示深度 4 时两端规模的原始梯度。判断“接近零”还要结合均值和 RMS：方差小但均值很大，表示各次初始化得到相似的非零梯度，不是 barren plateau。

In [ ]:
variance_plot(bp_summary)


In [ ]:
gradient_plot = vis.Plot(title="宽初始化的逐seed梯度", xlabel="seed序号", ylabel="∂C / ∂θ[0,0]")
for n, cost in ((2, "global"), (8, "global"), (8, "local")):
    selected = [r for r in records if r["kind"] == "pqc" and r["initialization"] == "wide"
                and r["qubits"] == n and r["depth"] == max(depths)]
    gradient_plot.scatter(range(len(selected)), [r[cost + "_gradient"] for r in selected], label=f"n={n}, {cost}")
gradient_plot


## 5. 一个有解析答案的对照

无纠缠电路中，`C_global=∏ cos(θᵢ)`，其首参数梯度为 `-sin(θ₀)∏ᵢ₌₁ cos(θᵢ)`。独立均匀采样时每个平方因子的期望是 `1/2`，所以方差为 `2⁻ⁿ`；局部 cost 的梯度始终是 `-sin(θ₀)`。

这个可检验的例子隔离了 cost 支撑范围的影响。它不是“所有局部 cost 都可训练”或“小角度必定修复平台”的结论。

In [ ]:
control_plot = vis.Plot(title="无纠缠对照：实测与解析方差", xlabel="量子比特数", ylabel="log10(方差)")
for cost in ("local", "global"):
    selected = [r for r in bp_summary if r["kind"] == "factorized" and r["cost"] == cost]
    control_plot.line([r["qubits"] for r in selected], [log10(max(r["variance"], 1e-30)) for r in selected], label=f"实测 {cost}")
control_plot.line(qubit_counts, [log10(0.5)] * len(qubit_counts), label="解析 local = 1/2")
control_plot.line(qubit_counts, [-n * log10(2) for n in qubit_counts], label="解析 global = 2^-n")
control_plot


## 如何继续实验

- 先把 `samples` 从 24 增到 100，观察区间是否收窄；再增加深度，避免同时改所有设置。
- 改一个 cost、纠缠拓扑或固定门结构时，保留相同参数采样 seed，检查选中参数确实影响读出。
- 同时记录均值、RMS、方差和可接受精度下的近零比例。有限精度的零不等同于数学上的严格零。
- 若要研究训练效果，再选固定任务比较实际优化曲线；本页测的是初始化梯度分布，没有宣称已解决量子 LM 的可训练性。

无需 PyTorch、NumPy 或其他量子 package。脚本模式可运行 `python examples/quantum/barren_plateau.py --device cuda --samples 24 --output runs/bp.json`；已有输出文件不会被覆盖。